In [ ]:
import requests
import json
import urllib.parse
from google.colab import userdata

# ==========================================
# CONFIGURATION
# ==========================================
API_KEY = userdata.get('KNOWLEDGEGRAPH_API_KEY')
ENTITY_ID = 'kg:/m/0gzkmh'
# ==========================================

In [ ]:
QUERY = 'Saxo Bank'

# The entity ID of Saxo Bank is kg:/m/0gzkmh
# Other entities showing up when searching for "Saxo Bank" are:
#
# E3 Saxo Bank Classic - kg:/m/08k2k7
# Team Saxo Bank-SunGard - kg:/m/06s1mb
#
#

In [ ]:
# 2. THE SEARCH FUNCTION USING A QUERY
# ==========================================
def search_knowledge_graph(api_key, query):
    service_url = 'https://kgsearch.googleapis.com/v1/entities:search'
    params = {
        'query': query,
        'limit': 5,
        'indent': True,
        'key': api_key,
    }

    url = service_url + '?' + urllib.parse.urlencode(params)

    try:
        response = requests.get(url)
        data = json.loads(response.text)

        print(f"--- 🔍 Knowledge Graph Results for: '{query}' ---\n")

        if 'itemListElement' not in data or len(data['itemListElement']) == 0:
            print("No Knowledge Graph entry found. Google may not see this as a distinct entity yet.")
            return

        for element in data['itemListElement']:
            result = element.get('result', {})
            score = element.get('resultScore', 0)

            # --- Basic Info ---
            name = result.get('name', 'Unknown Name')
            entity_id = result.get('@id', 'N/A')
            types = result.get('@type', [])

            # --- Description & Bio ---
            short_desc = result.get('description', 'No short description.')

            # detailedDescription is a nested object
            detailed_obj = result.get('detailedDescription', {})
            bio_body = detailed_obj.get('articleBody', 'No detailed bio available.')
            bio_url = detailed_obj.get('url', 'No bio source URL.')

            # --- Visuals ---
            # Image is often a nested object
            image_obj = result.get('image', {})
            image_url = image_obj.get('contentUrl', 'No image available.')

            # --- Official Web Link ---
            website = result.get('url', 'No official website listed.')

            # --- PRINTING THE REPORT ---
            print(f"Name:           {name}")
            print(f"Relevance:      {score}")
            print(f"Entity ID:      {entity_id}")
            print(f"Types:          {', '.join(types)}")
            print(f"Short Desc:     {short_desc}")
            print(f"Official Web:   {website}")
            print(f"Image URL:      {image_url}")
            print(f"Bio Source:     {bio_url}")
            print(f"Bio Body:       {bio_body[:200]}..." if len(bio_body) > 200 else f"Bio Body:       {bio_body}")
            print("-" * 50)

    except Exception as e:
        print(f"Error: {e}")

# ==========================================
# 3. RUN IT
# ==========================================
if __name__ == "__main__":
    search_knowledge_graph(API_KEY, QUERY)

--- 🔍 Knowledge Graph Results for: 'Saxo Bank' ---

Name:           Saxo Bank
Relevance:      7784.4755859375
Entity ID:      kg:/m/0gzkmh
Types:          Organization, Thing, Corporation
Short Desc:     Investment banking company
Official Web:   http://www.saxobank.com
Image URL:      https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcQ6gxd82SwUaHfvJPFpHEX0ROgg9OaRLLMvuN8zFIK57QV9CvBw
Bio Source:     https://en.wikipedia.org/wiki/Saxo_Bank
Bio Body:       Saxo Bank is a Danish investment bank specializing in online trading and investment. Established in 1992 as a brokerage firm under the name Midas Fondsmæglerselskab by Lars Seier Christensen and Kim F...
--------------------------------------------------
Name:           E3 Saxo Bank Classic
Relevance:      540.0468139648438
Entity ID:      kg:/m/08k2k7
Types:          Thing
Short Desc:     No short description.
Official Web:   No official website listed.
Image URL:      No image available.
Bio Source:     https://en.wikipedia.org/

In [ ]:
# ==========================================
# 2. THE ID LOOKUP FUNCTION
# ==========================================
def lookup_entity_by_id(api_key, entity_id):
    service_url = 'https://kgsearch.googleapis.com/v1/entities:search'

    # The API technically prefers IDs without the 'kg:' prefix for direct lookup
    # We will try the user's input first, and if it fails, we try stripping 'kg:'
    ids_to_try = [entity_id]
    if entity_id.startswith('kg:'):
        ids_to_try.append(entity_id.replace('kg:', ''))

    for current_id in ids_to_try:
        params = {
            'ids': current_id, # <--- We use 'ids' instead of 'query'
            'indent': True,
            'key': api_key,
        }

        url = service_url + '?' + urllib.parse.urlencode(params)

        try:
            response = requests.get(url)
            data = json.loads(response.text)

            # If we find a result, break the loop and print it
            if 'itemListElement' in data and len(data['itemListElement']) > 0:
                print(f"--- ✅ Success! Found entry for ID: {current_id} ---\n")

                element = data['itemListElement'][0] # We only expect one result for an ID
                result = element.get('result', {})
                score = element.get('resultScore', 0)

                name = result.get('name', 'Unknown Name')
                types = result.get('@type', [])
                desc = result.get('description', 'No description.')
                website = result.get('url', 'No website.')
                detailed = result.get('detailedDescription', {}).get('articleBody', 'No bio.')

                print(f"Name:        {name}")
                print(f"Relevance:   {score}")
                print(f"Types:       {', '.join(types)}")
                print(f"Description: {desc}")
                print(f"Website:     {website}")
                print(f"Bio:         {detailed[:200]}..." if len(detailed) > 200 else f"Bio: {detailed}")
                return # Exit function after success

        except Exception as e:
            print(f"Error connecting to API: {e}")

    # If the loop finishes without returning, we found nothing
    print(f"❌ Could not find any entity for ID: {entity_id}")
    print("Tip: Ensure the ID is exactly as it appeared in the previous search.")

# ==========================================
# 3. RUN IT
# ==========================================
if __name__ == "__main__":
    lookup_entity_by_id(API_KEY, ENTITY_ID)

--- ✅ Success! Found entry for ID: /m/0gzkmh ---

Name:        Saxo Bank
Relevance:   0
Types:       Corporation, Thing, Organization
Description: Investment banking company
Website:     http://www.saxobank.com
Bio:         Saxo Bank is a Danish investment bank specializing in online trading and investment. Established in 1992 as a brokerage firm under the name Midas Fondsmæglerselskab by Lars Seier Christensen and Kim F...


In [ ]:
Simport requests
import json
import urllib.parse
from google.colab import userdata

# ==========================================
# RAW DATA DUMP
# ==========================================
def get_raw_entity_data(api_key, entity_id):
    service_url = 'https://kgsearch.googleapis.com/v1/entities:search'

    # The API technically prefers IDs without the 'kg:' prefix for direct lookup
    # We will try the user's input first, and if it fails, we try stripping 'kg:'
    ids_to_try = [entity_id]
    if entity_id.startswith('kg:'):
        ids_to_try.append(entity_id.replace('kg:', ''))

    for current_id in ids_to_try:
        params = {
            'ids': current_id,
            'indent': True,
            'key': api_key,
        }

        url = service_url + '?' + urllib.parse.urlencode(params)

        try:
            response = requests.get(url)
            data = json.loads(response.text)

            # We print the raw JSON of the first result found
            if 'itemListElement' in data and len(data['itemListElement']) > 0:
                result_raw = data['itemListElement'][0]
                print(f"--- 📂 Raw Data for {current_id} ---\n")
                print(json.dumps(result_raw, indent=4))
                return # Exit function after success

        except Exception as e:
            print(f"Error: {e}")

    # If the loop finishes without returning, we found nothing
    print(f"No data found for ID: {entity_id}.")

if __name__ == "__main__":
    get_raw_entity_data(API_KEY, ENTITY_ID)

--- 📂 Raw Data for /m/0gzkmh ---

{
    "resultScore": 0,
    "@type": "EntitySearchResult",
    "result": {
        "@id": "kg:/m/0gzkmh",
        "description": "Investment banking company",
        "@type": [
            "Organization",
            "Thing",
            "Corporation"
        ],
        "detailedDescription": {
            "articleBody": "Saxo Bank is a Danish investment bank specializing in online trading and investment. Established in 1992 as a brokerage firm under the name Midas Fondsm\u00e6glerselskab by Lars Seier Christensen and Kim Fournais, the company rebranded as Saxo Bank in 2001 upon obtaining its banking license. ",
            "license": "https://en.wikipedia.org/wiki/Wikipedia:Text_of_Creative_Commons_Attribution-ShareAlike_3.0_Unported_License",
            "url": "https://en.wikipedia.org/wiki/Saxo_Bank"
        },
        "image": {
            "url": "https://ru.wikipedia.org/wiki/%D0%A4%D0%B0%D0%B9%D0%BB:Saxobank-logo-3.png",
            "contentUr